# 00 — Verify LIBERO-PRO (assets, episode identity, one smoke rollout)

Throwaway verification notebook. Answers three questions before any real collection:

1. **Do all 16 PRO suites instantiate?** (exercises the two env patches the package is missing)
2. **Does the package load the *same episodes* as the prior 16-suite run?** (hash-level proof, no rollouts)
3. **Does a rollout complete end to end and land in Supabase?**

Phase A needs no policy load and no GPU rollouts. Run it first — it answers (1) and (2) in ~5 minutes.

Assets come from **your existing Drive tarball**, not from `install_assets`: the package has no
HuggingFace path, so it cannot provision the `_swap` / `_task` suites. Using the same tarball the
prior run used is also what makes the episode-identity check in A6b meaningful.

## Before running

- **Fresh Colab GPU runtime** (L4 is enough). `setup_environment()` raises on a CPU runtime, so
  even Phase A needs the GPU.
- **Colab Secrets, all four, with notebook access granted:** `GH_PAT`, `HF_TOKEN`, `SUPABASE_URL`,
  `SUPABASE_SERVICE_KEY`. Cell A1 reads all of them, so a missing one fails immediately.
- The notebook clones `main` and `pip install -e`s the package, so it does **not** need to live in
  the repo — uploading it to Drive and opening it in Colab is fine.
- Check the Drive paths in A2 against your layout.

---
## Phase A — assets + manifest (no policy load)
### A1. Bootstrap

In [ ]:
import os
import subprocess
import sys
from google.colab import userdata

for key in ("SUPABASE_URL", "SUPABASE_SERVICE_KEY", "HF_TOKEN"):
    os.environ[key] = userdata.get(key)

gh_pat = userdata.get("GH_PAT")
repo_dir = "/content/cs159-sp26"
repo_url = f"https://{gh_pat}@github.com/ArjunS07/cs159-sp26.git"

if not os.path.isdir(os.path.join(repo_dir, ".git")):
    subprocess.run(["git", "clone", "--branch", "main", repo_url, repo_dir], check=True)
else:
    subprocess.run(["git", "-C", repo_dir, "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", repo_dir, "checkout", "main"], check=True)
    subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only", "origin", "main"], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-e", f"{repo_dir}/pnp-vla[sim]",
], check=True)

# Editable installs add a .pth file that a fresh interpreter would process at startup. This
# notebook interpreter was already running during pip, so expose the source tree immediately.
package_dir = f"{repo_dir}/pnp-vla"
if package_dir not in sys.path:
    sys.path.insert(0, package_dir)

import pnp
print("Loaded pnp from:", pnp.__file__)

### A1b. System GL/EGL libraries (once per runtime)

`setup_environment()` sets `MUJOCO_GL=egl`, and A7 renders offscreen. The refactored worker
notebooks omit this cell and the stock-LIBERO collection completed without it, so recent Colab
images evidently ship what MuJoCo needs — but it is ~30s of insurance against a whole class of
`EGL`/`GLEW`/`libGL` failures at env creation. Skip it if you would rather find out.

(Ported from `pnp_pro_experiment_averages` cell 4.)

In [ ]:
%%bash
apt-get update -qq
apt-get install -y -qq \
    libosmesa6-dev libgl1-mesa-glx libglfw3 libglew-dev \
    libegl1-mesa-dev patchelf ffmpeg
mkdir -p /usr/share/glvnd/egl_vendor.d
echo '{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}' \
    > /usr/share/glvnd/egl_vendor.d/10_nvidia.json
echo 'System GL deps done'

### A1c. Environment validation

**Requires a GPU runtime** — `setup_environment()` raises if `torch.cuda.is_available()` is false,
so Phase A cannot run on CPU. It also pins `HF_HOME` to local disk (`/content/hf_home`), never
Drive, so the pi0.5 checkpoint re-downloads once per runtime in Phase B.

In [ ]:
from pnp.env_setup import setup_environment
setup_environment()  # If the runtime restarts, use Run all again.

### A2. Config — Drive paths

`ASSET_TAR` is the tarball the prior 16-suite run used. `LEGACY_DB` is the corrected-baseline
SQLite DB; A6b skips itself if the file is absent.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE       = '/content/drive/MyDrive'
CACHE_DIR   = f'{DRIVE}/cs159_jeff/smolvla_colab_cache'
RESULTS_DIR = f'{DRIVE}/cs159_jeff/libero_pro_results'

ASSET_TAR = f'{CACHE_DIR}/libero_pro_assets_enlarged_hf.tar.gz'
LEGACY_DB = f'{RESULTS_DIR}/rollouts_enlarged_uncertainty_fix.db'

EPISODES_PER_TASK = 25          # the prior run's cap; the package default is 10
SMOKE_EXPERIMENT  = 'pro-smoke-v1'

assert os.path.exists(ASSET_TAR), f'asset tarball not found: {ASSET_TAR}'
print(f'asset tarball : {ASSET_TAR}')
print(f'legacy DB     : {LEGACY_DB}  ({"present" if os.path.exists(LEGACY_DB) else "MISSING -> A6b will skip"})')

### A3. Clone LIBERO-PRO, restore the asset tarball

The clone supplies the three patched `libero` source files (and `notebooks/custom_assets`); the
tarball supplies the bddl + init files for all 16 suites. `install_assets()` is deliberately **not**
called — it reads the clone only and would raise `FileNotFoundError` on the `_swap` / `_task` suites.

In [ ]:
import sys, tarfile
import torch
from pnp import libero_pro

PRO_DIR     = libero_pro.clone_libero_pro()
LIBERO_SITE = libero_pro.libero_site()
print(f'clone       : {PRO_DIR} @ {libero_pro.LIBERO_PRO_REVISION[:12]}')
print(f'libero site : {LIBERO_SITE}')

with tarfile.open(ASSET_TAR, 'r:gz') as tar:
    tar.extractall(path=LIBERO_SITE)   # merges bddl_files/<suite> and init_files/<suite>
print(f'restored {ASSET_TAR}')

SUITES = list(libero_pro.EXPANDED_PRO_SUITES)
assert len(SUITES) == 16, len(SUITES)


def states_per_task(suite):
    d = os.path.join(LIBERO_SITE, 'init_files', suite)
    files = sorted(f for f in os.listdir(d) if f.endswith('.pruned_init')) if os.path.isdir(d) else []
    if not files:
        return 'MISSING', 0
    try:
        return len(torch.load(os.path.join(d, files[0]), weights_only=False)), len(files)
    except Exception as exc:
        return f'err {exc}', len(files)


print(f'\n{"suite":<36}{"states/task":>12}{"init files":>12}')
print('-' * 60)
missing = []
for suite in SUITES:
    n_states, n_files = states_per_task(suite)
    flag = '  <-- MISSING' if n_states == 'MISSING' else ''
    print(f'{suite:<36}{str(n_states):>12}{n_files:>12}{flag}')
    if n_states == 'MISSING':
        missing.append(suite)
assert not missing, f'{len(missing)} suites have no init files: {missing}'
print(f'\nOK: all {len(SUITES)} suites have init files.')

### A4. Env patches + the two the package is missing

`apply_env_patches` must run **after** the tarball restore (it scans `bddl_files/` to register
suites) and `reload_benchmark` **after** it. The two inline blocks below are the gaps found in the
audit — kept in the notebook rather than the package until this run says they are needed.

In [ ]:
libero_pro.apply_env_patches(site=LIBERO_SITE, pro_dir=PRO_DIR)
libero_pro.patch_torch_load()

In [ ]:
# ── GAP 1: custom_assets for the distractor (_with_*) suites ─────────────────
# Ported from pnp_pro_experiment_averages cell 17 Step 6. No counterpart in pnp/.
import shutil

REPO_ASSETS = os.path.join(PRO_DIR, 'notebooks', 'custom_assets')
DIST_ASSETS = f'/usr/local/lib/python3.{sys.version_info.minor}/dist-packages/notebooks/custom_assets'
if not os.path.isdir(REPO_ASSETS):
    print(f'NOTE: {REPO_ASSETS} not in the clone -- nothing to copy')
elif os.path.exists(DIST_ASSETS):
    print(f'custom_assets already present: {DIST_ASSETS}')
else:
    shutil.copytree(REPO_ASSETS, DIST_ASSETS)
    print(f'copied custom_assets -> {DIST_ASSETS} ({len(os.listdir(DIST_ASSETS))} entries)')

In [ ]:
# ── GAP 2: OBJECTS_DICT aliases ──────────────────────────────────────────────
# Ported from pnp_pro_experiment_averages cell 29. Registered *after* apply_env_patches copies
# envs/objects/__init__.py -- copying that file alone was not sufficient in the notebook.
from libero.libero.envs.objects import OBJECTS_DICT

ALIASES = {
    'black_bowl':              'akita_black_bowl',
    'yellow_plate':            'plate',
    'bigger_akita_black_bowl': 'akita_black_bowl',
    'brown_rack':              'wine_rack',
    'red_cream_cheese':        'cream_cheese',
    'white_bottle':            'wine_bottle',
    'yellow_cabinet':          'wooden_cabinet',
    'yellow_stove':            'stove',
}
added = []
for alias, base in ALIASES.items():
    if alias not in OBJECTS_DICT and base in OBJECTS_DICT:
        OBJECTS_DICT[alias] = OBJECTS_DICT[base]
        added.append(alias)
print(f'OBJECTS_DICT: {len(OBJECTS_DICT)} entries; registered {len(added)} alias(es): {added}')
unresolved = [a for a, b in ALIASES.items() if a not in OBJECTS_DICT]
print(f'unresolved aliases (base name also absent): {unresolved}')

In [ ]:
bd = libero_pro.reload_benchmark()
absent = [s for s in SUITES if s not in bd]
assert not absent, f'not registered in the benchmark dict: {absent}'
print(f'all {len(SUITES)} suites registered.')

### A5. Build the episode manifest

`EXPANDED_PRO_SUITES` is element-for-element equal to the prior run's `SUITES`. Passing
`episode_idxs=range(25)` restores the prior run's 25 episodes/task (the package default is 10).

In [ ]:
import collections

episodes = libero_pro.build_libero_pro_episodes(
    bd, suites=SUITES, episode_idxs=range(EPISODES_PER_TASK))

by_suite = collections.Counter(ep['suite'] for ep in episodes)
tasks_by_suite = {s: len({ep['task_idx'] for ep in episodes if ep['suite'] == s}) for s in SUITES}
steps_by_suite = {s: sorted({ep['max_steps'] for ep in episodes if ep['suite'] == s}) for s in SUITES}

print(f'{"suite":<36}{"tasks":>6}{"episodes":>10}{"max_steps":>11}')
print('-' * 63)
for suite in SUITES:
    print(f'{suite:<36}{tasks_by_suite[suite]:>6}{by_suite[suite]:>10}'
          f'{str(steps_by_suite[suite]):>11}')
print(f'\nTOTAL {len(episodes)} episodes across {len(SUITES)} suites')

identities = {(ep['suite'], ep['task_idx'], ep['ep_idx'], ep['init_state_hash']) for ep in episodes}
assert len(identities) == len(episodes), 'duplicate identity in the manifest'
print(f'{len(identities)} unique identities (no duplicates).')

### A5b. `max_steps` vs the LIBERO-PRO spec

LIBERO-PRO's `TASK_MAX_STEPS` gives every perturbed suite its base suite's canonical limit —
goal 300, spatial 220, `libero_10` 520, object 280. The package sends every PRO suite to 280
(`libero_pro.py` `MAX_STEPS_MAP.get(suite, LIBERO_PRO_MAX_STEPS)`). This cell only **reports** the
gap; it does not change behavior.

In [ ]:
from pnp.config import MAX_STEPS_MAP

BASE_PREFIXES = ('libero_10', 'libero_90', 'libero_goal', 'libero_object', 'libero_spatial')


def upstream_max_steps(suite):
    """LIBERO-PRO convention: a perturbed suite inherits its base suite's canonical limit."""
    for prefix in sorted(BASE_PREFIXES, key=len, reverse=True):
        if suite.startswith(prefix):
            return MAX_STEPS_MAP[prefix]
    return None


print(f'{"suite":<36}{"current":>9}{"upstream":>10}{"delta":>8}')
print('-' * 63)
mismatched = []
for suite in SUITES:
    current = steps_by_suite[suite][0]
    upstream = upstream_max_steps(suite)
    delta = '' if current == upstream else f'{upstream - current:+d}'
    if current != upstream:
        mismatched.append((suite, current, upstream))
    print(f'{suite:<36}{current:>9}{upstream:>10}{delta:>8}')
print(f'\n{len(mismatched)}/{len(SUITES)} suites differ from the LIBERO-PRO spec.')
print('Fix before real collection (it moves SR); harmless for this smoke test.')

### A6a. Init-state path equivalence

The package subclasses LIBERO's own `Benchmark` and inherits `get_task_init_states`, where the
prior notebook hand-rolled `init_files/<suite>/<task>.pruned_init`. This asserts the two resolve to
the same array. A mismatch here is the failure mode that inflated `temp_x0.1` from ~45% to ~90%.

In [ ]:
import numpy as np

CHECK_SUITES = ['libero_object_temp_x0.1', 'libero_goal_swap', 'libero_spatial_with_milk']

for suite in CHECK_SUITES:
    task_suite = bd[suite]()
    for task_idx in range(task_suite.n_tasks):
        task = task_suite.get_task(task_idx)
        via_libero = np.asarray(task_suite.get_task_init_states(task_idx))
        explicit_path = os.path.join(
            LIBERO_SITE, 'init_files', suite,
            task.bddl_file.replace('.bddl', '.pruned_init'))
        assert os.path.exists(explicit_path), f'{suite} t{task_idx}: no {explicit_path}'
        via_explicit = np.asarray(torch.load(explicit_path, weights_only=False))
        assert np.array_equal(via_libero, via_explicit), (
            f'{suite} t{task_idx}: LIBERO resolved a DIFFERENT init array than '
            f'{explicit_path} (shapes {via_libero.shape} vs {via_explicit.shape})')
    print(f'{suite:<36} OK  ({task_suite.n_tasks} tasks, '
          f'problem_folder={task_suite.get_task(0).problem_folder})')
print('\nInit-state resolution matches the explicit per-suite path.')

### A6b. Episode identity vs the prior run

Both harnesses take a **prefix** of the same init arrays (the notebook capped to the first 25; this
manifest indexes 0..24), so `init_state_hash` must match episode-for-episode. The package uses
`md5[:12]`, the notebook `md5[:16]` — a strict prefix, so the legacy values are truncated to 12.

Matching hashes prove the refactored path loads exactly the prior run's episodes, without running
a single rollout.

In [ ]:
import sqlite3

if not os.path.exists(LEGACY_DB):
    print(f'SKIPPED: {LEGACY_DB} not found.')
else:
    con = sqlite3.connect(f'file:{LEGACY_DB}?mode=ro', uri=True)
    legacy = {
        (suite, int(task_idx), int(ep_idx)): h[:12]
        for suite, task_idx, ep_idx, h in con.execute(
            "SELECT suite, task_idx, episode_idx, init_state_hash FROM rollouts "
            "WHERE pnp_mode='uncertainty'")
    }
    con.close()
    print(f'legacy uncertainty rows: {len(legacy)} unique (suite, task, episode) keys\n')

    new = {(ep['suite'], ep['task_idx'], ep['ep_idx']): ep['init_state_hash'] for ep in episodes}
    overlap = sorted(set(new) & set(legacy))
    agree = [k for k in overlap if new[k] == legacy[k]]
    disagree = [k for k in overlap if new[k] != legacy[k]]

    print(f'{"suite":<36}{"compared":>10}{"agree":>8}{"DISAGREE":>10}{"new-only":>10}')
    print('-' * 76)
    for suite in SUITES:
        cmp_n = sum(1 for k in overlap if k[0] == suite)
        ok_n = sum(1 for k in agree if k[0] == suite)
        bad_n = sum(1 for k in disagree if k[0] == suite)
        only_n = sum(1 for k in new if k[0] == suite and k not in legacy)
        print(f'{suite:<36}{cmp_n:>10}{ok_n:>8}{bad_n:>10}{only_n:>10}')

    print(f'\ncompared {len(overlap)}  agree {len(agree)}  DISAGREE {len(disagree)}')
    if disagree:
        print('\nFirst 10 mismatches (suite, task, episode): new != legacy')
        for k in disagree[:10]:
            print(f'  {k}: {new[k]} != {legacy[k]}')
    assert not disagree, (
        f'{len(disagree)} episodes resolve to a DIFFERENT init state than the prior run -- '
        'the asset source is not the corrected one. Do not collect.')
    print('\nEvery overlapping episode loads the SAME init state as the prior run.')

### A7. Instantiate every suite

This is what actually exercises the two patches from A4: bddl parsing resolves object names through
`OBJECTS_DICT`, and MuJoCo loads the distractor meshes. One env per suite (task 0), reset +
`set_init_state` + the standard settling steps, then closed.

In [ ]:
from pnp.config import LIBERO_DUMMY_ACTION, NUM_STEPS_WAIT
from pnp import libero_env

first_ep = {}
for ep in episodes:
    first_ep.setdefault(ep['suite'], ep)

failures = {}
for suite in SUITES:
    ep = first_ep[suite]
    env = None
    try:
        env = libero_env.make_env(ep['bddl_path'])
        env.reset()
        obs = env.set_init_state(ep['init_state'])
        for _ in range(NUM_STEPS_WAIT):
            obs, _, _, _ = env.step(LIBERO_DUMMY_ACTION)
        shape = obs['agentview_image'].shape
        print(f'{suite:<36} OK   agentview={shape}')
    except Exception as exc:
        failures[suite] = f'{type(exc).__name__}: {exc}'
        print(f'{suite:<36} FAIL {type(exc).__name__}: {exc}')
    finally:
        if env is not None:
            env.close()

print(f'\n{len(SUITES) - len(failures)}/{len(SUITES)} suites instantiate.')
if failures:
    print('\nFailures:')
    for suite, msg in failures.items():
        print(f'  {suite}: {msg}')
    print('\nIf a distractor (_with_*) suite failed, A4 gap 1 is real.')
    print('If the error is a KeyError on an object name, A4 gap 2 is real (add that alias).')
assert not failures, f'{len(failures)} suites failed to instantiate'

**Phase A done.** If A6a/A6b/A7 all passed, episode loading is identical to the prior run and all
16 suites are reachable. Phase B below needs a GPU and loads the policy.

---
## Phase B — rollout smoke test
### B1. Load the policy

In [ ]:
from pnp import models

policy, preprocess, postprocess = models.load_pi05()
device = models.default_device()
print(f'device={device}  chunk_size={policy.config.chunk_size}  '
      f'num_inference_steps={policy.config.num_inference_steps}')

### B2. P&P no-op / RNG-isolation contract

`assert_pnp_noop` checks that a measurement-only probe (a) leaves the global RNG state
bit-identical to a vanilla chunk and (b) returns the same action within the bf16 nondeterminism
floor. Nothing in the repo calls it, so run it once here.

In [ ]:
from pnp.libero_env import obs_to_policy
from pnp.pnp import assert_pnp_noop

probe_ep = first_ep['libero_object_temp_x0.1']
env = libero_env.make_env(probe_ep['bddl_path'])
try:
    env.reset()
    obs = env.set_init_state(probe_ep['init_state'])
    for _ in range(NUM_STEPS_WAIT):
        obs, _, _, _ = env.step(LIBERO_DUMMY_ACTION)
    batch = preprocess(obs_to_policy(obs, probe_ep['task_desc']))
finally:
    env.close()

assert_pnp_noop(policy, batch, step_indices=(4, 5))

### B3. One identity per perturbation family × 3 PRO configs

`build_pro_methods()` is what the six PRO workers run: the observed/no-op arm at steps 1–9 with
PCP features, a 16-step matched-compute control, and refine-last `(4,5)`. 4 identities × 3 configs
= 12 rollouts under a throwaway experiment label.

In [ ]:
from tqdm.auto import tqdm

from pnp.experiments import build_pro_methods
from pnp.rollout import iter_task_envs, run_episode
from pnp.store import SupabaseStore

SMOKE_SUITES = [
    'libero_object_temp_x0.1',      # position perturbation (repo-sourced init)
    'libero_goal_swap',             # swap (HF-sourced init)
    'libero_goal_task',             # task redefinition (HF-sourced init)
    'libero_spatial_with_milk',     # distractor (needs custom_assets)
]
smoke_episodes = [ep for ep in episodes
                  if ep['suite'] in SMOKE_SUITES and ep['task_idx'] == 0 and ep['ep_idx'] == 0]
methods = build_pro_methods()
assert len(smoke_episodes) == len(SMOKE_SUITES), smoke_episodes

print(f'{len(smoke_episodes)} identities x {len(methods)} configs = '
      f'{len(smoke_episodes) * len(methods)} rollouts')
for name, cfg in methods:
    print(f'  {name:<24} pnp_steps={cfg.pnp_steps} k={cfg.pnp_k} refine={cfg.refine} '
          f'steps={cfg.num_inference_steps} pcp={cfg.save_pcp_features}')

In [ ]:
store = SupabaseStore()
store.start_run(
    driver='pro_smoke_verify', benchmark='libero_pro', experiment=SMOKE_EXPERIMENT,
    notes='asset/episode/rollout verification of the refactored LIBERO-PRO path',
    config={'suites': SMOKE_SUITES, 'episodes_per_task': EPISODES_PER_TASK,
            'libero_pro_revision': libero_pro.LIBERO_PRO_REVISION,
            'patched_custom_assets': True, 'patched_objects_dict_aliases': True},
)

done = store.existing_keys(SMOKE_EXPERIMENT)
results, n_logged = [], 0
for env, task_eps in iter_task_envs(smoke_episodes):
    for ep, name, cfg, rid in tqdm(list(store.iter_todo(SMOKE_EXPERIMENT, task_eps, methods, done)),
                                   desc=f"{task_eps[0]['suite']} t{task_eps[0]['task_idx']}"):
        result = run_episode(env, ep, policy, preprocess, postprocess, device, cfg)
        store.log_result(rid, ep, name, cfg, result)
        n_logged += 1
        results.append({'suite': ep['suite'], 'method': name, 'status': result['status'],
                        'success': result['success'], 'n_steps': result['n_steps'],
                        'max_steps': ep['max_steps'], 'elapsed_s': round(result['elapsed_s'], 1),
                        'terminated_reason': result['terminated_reason'],
                        'error_msg': result['error_msg']})
store.finish_run(n_rollouts=n_logged)
print(f'logged {n_logged} rollouts')

### B4. Verify what landed

`run_episode` catches every exception into `status='errored'`, so a loop that looked clean proves
nothing — read the column back from Supabase.

In [ ]:
import pandas as pd

local = pd.DataFrame(results)
print('=== local results ===')
print(local.to_string(index=False))

rows = store.client.table('rollouts').select(
    'suite,method,status,success,n_steps,max_steps,terminated_reason,error_msg,'
    'u_mean_episode,n_pnp_activations,n_vf_evals,pcp_chunks_path'
).eq('experiment', SMOKE_EXPERIMENT).execute().data
remote = pd.DataFrame(rows)
print('\n=== supabase rollouts ===')
print(remote.to_string(index=False))

errored = remote[remote.status != 'completed']
assert remote.shape[0] == len(smoke_episodes) * len(methods), (
    f'expected {len(smoke_episodes) * len(methods)} rows, found {remote.shape[0]}')
assert errored.empty, f'{len(errored)} errored rollouts:\n{errored.to_string(index=False)}'

observed = remote[remote.method == 'pnp_uncertainty_only']
assert (observed.n_pnp_activations > 0).all(), 'observed arm recorded no uncertainty steps'
assert observed.pcp_chunks_path.notna().all(), 'observed arm wrote no pcp_chunks blob'
print('\nAll rollouts completed; observed arm recorded uncertainty + PCP features.')

---
## Result

| Check | Meaning |
|---|---|
| A3 | all 16 suites have init files |
| A5 | manifest builds, no duplicate identities |
| A5b | reports the `max_steps` gap vs the LIBERO-PRO spec (fix before real collection) |
| A6a | LIBERO resolves the same init array as the explicit per-suite path |
| A6b | every overlapping episode loads the prior run's init state |
| A7 | all 16 suites instantiate in MuJoCo |
| B2 | the probe is a true no-op and does not advance the global RNG |
| B3/B4 | 12 rollouts complete and land in Supabase with telemetry |

Then, before real collection: fix `max_steps`; fold the A4 patches into
`libero_pro.apply_env_patches` if A7 needed them; decide canonical-6 vs the 16-suite union (the
union needs a HuggingFace branch in `install_assets`, and `analysis/validate.py` currently
hard-asserts the canonical 600).

Clean up this smoke experiment when done:

```python
store.client.table('rollouts').delete().eq('experiment', SMOKE_EXPERIMENT).execute()
```